# 8.1 · Bagging / Bootstrap Aggregating

> **课程定位 / Where this fits**
> 第 1 课，**Part 8 · 集成学习**。
> Lesson 1, **Part 8 · Ensemble Learning**.
>
> 集成学习有两大范式：**bagging（降方差）** 和 **boosting（降偏差）**。这一课从头讲 bagging——它把同一种模型在**不同 bootstrap 样本**上各训一个，再**平均/投票**，从而把"高方差模型"的随机抖动抵消掉。随机森林(5.7/8.2)就是 bagging 的明星应用，但 bagging 是一个**通用框架**，可以套在任何模型上。
> Ensemble learning has two paradigms: **bagging (reduces variance)** and **boosting (reduces bias)**. This lesson builds bagging from scratch — it trains the same model on **different bootstrap samples** and **averages/votes**, canceling the random wobble of high-variance models. Random Forest (5.7/8.2) is bagging's star application, but bagging is a **general framework** wrappable around any model.
>
> 💼 **实战/面试视角**："bagging 怎么降方差 / bootstrap 是什么 / OOB / bagging vs boosting" 是集成必考。
> 💼 **Practical/interview angle:** "how bagging reduces variance / bootstrap / OOB / bagging vs boosting" — ensemble must-knows.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $B$ —— 基学习器个数 / number of base learners
> - bootstrap —— 有放回抽样得到的子集（约 63% 不重复样本）/ a with-replacement subset
> - $\rho$ —— 基学习器预测之间的相关系数 / correlation between learners' predictions

> 💡 **面试相关 / Interview-relevant**
> - "bagging 降方差的数学原理"（出镜率 ★★★★★）
> - "bootstrap 为什么约覆盖 63% 样本"（★★★★）
> - "OOB 误差是什么"（★★★★）
> - "bagging 适合什么基学习器（高方差/不稳定）"（★★★★）
> - "bagging vs boosting"（★★★★★）

---

## 学习目标 / Learning Objectives

1. 理解 bootstrap 抽样 + 约 63% 覆盖率。
   Understand bootstrap sampling and the ~63% coverage.
2. 用数学解释 bagging 为何降方差。
   Explain mathematically why bagging reduces variance.
3. **从零**实现 bagging 分类器并对照 sklearn。
   Implement a bagging classifier from scratch and match sklearn.
4. 理解 bagging **只对高方差/不稳定模型有效**。
   Understand bagging helps only high-variance/unstable learners.
5. 用 **OOB** 做免费验证。
   Use OOB for free validation.

## 目录 / TOC
1. [先建直觉 + bootstrap ⭐](#1)
2. [bagging 降方差的数学 ⭐](#2)
3. [🚢 数据 + 从零实现 ⭐](#3)
4. [谁适合 bagging：高方差才有效 ⭐](#4)
5. [OOB 免费验证 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉 + bootstrap ⭐ / Intuition & Bootstrap

bagging 的核心思想是"**群体智慧**"：一个高方差模型（如不剪枝的决策树）对训练数据的随机扰动很敏感，换一批数据就给出很不同的预测。但如果我们训练**很多个**这样的模型、让它们**投票/平均**，每个模型的随机误差会**互相抵消**，集体预测就稳得多。
The core idea of bagging is "**wisdom of crowds**": a high-variance model (e.g. an unpruned tree) is sensitive to training-data noise and gives very different predictions on a different sample. But if we train **many** such models and let them **vote/average**, their random errors **cancel out**, and the collective is far steadier.

怎么造出"很多个略有差异的训练集"？用 **bootstrap（自助抽样）**：从原训练集里**有放回地**随机抽 n 个样本（n = 原集大小）。因为有放回，有的样本被抽中多次、有的一次没抽中。一个经典结论（面试常考）：当 n 较大时，**每个 bootstrap 样本平均覆盖约 63.2% 的原始样本**，剩下约 36.8% 叫"袋外(OOB)"。
How to make "many slightly-different training sets"? Use the **bootstrap**: sample n points **with replacement** from the original training set (n = its size). With replacement, some points appear multiple times, others not at all. A classic result (often asked): for large n, **each bootstrap covers ~63.2% of the originals on average**, leaving ~36.8% "out-of-bag (OOB)".


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 验证 ~63% 覆盖率: 有放回抽 n 次, 看覆盖了多少不同样本 / verify the ~63% rule
print("bootstrap 覆盖率(有放回抽 n 个, 去重后占比):")
for n in [10, 100, 1000, 10000]:
    coverage = [len(np.unique(rng.integers(0, n, n)))/n for _ in range(200)]  # 200 次取平均
    print(f"  n={n:<6}: 平均覆盖 {np.mean(coverage):.1%}")
print(f"\n理论值 = 1 - (1-1/n)^n → 1 - 1/e = {1-1/np.e:.1%}  (n 越大越接近 63.2%)")
print("剩下约 36.8% 没被抽中 = 袋外(OOB)样本, 可做免费验证(第5节)")


<a id="2"></a>
## 2. bagging 降方差的数学 ⭐ / The Math of Variance Reduction

为什么"平均一堆模型"能降方差？设 $B$ 个基学习器的预测各自方差为 $\sigma^2$、两两相关系数为 $\rho$，它们平均后的方差是（面试核心公式）：
Why does averaging models reduce variance? With $B$ learners each of variance $\sigma^2$ and pairwise correlation $\rho$, the variance of their average is (a core interview formula):

$$\text{Var}\Big(\tfrac1B\sum_i\hat f_i\Big) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$$

- 第二项 $\frac{1-\rho}{B}\sigma^2$ 随 $B\to\infty$ **趋于 0**——加越多模型，这部分方差越小。
  The second term $\frac{1-\rho}{B}\sigma^2$ **goes to 0** as $B\to\infty$ — more models shrink this part.
- 但第一项 $\rho\sigma^2$ **不随 B 变**——它是方差的下限，由**模型之间的相关性 $\rho$** 决定。
  But the first term $\rho\sigma^2$ **doesn't depend on B** — it's the variance floor, set by the **correlation $\rho$** between models.

**两个关键推论**：(1) 偏差几乎不变（每个模型偏差相同，平均后还是它），bagging **只降方差不降偏差**；(2) 光加模型不够，**还要让模型彼此去相关（降 $\rho$）**——这正是随机森林在 bagging 基础上再加"特征随机"的原因(8.2)。
**Two key corollaries:** (1) bias is ~unchanged (same bias for each, averaging keeps it), so bagging **reduces variance, not bias**; (2) adding models isn't enough — you must also **de-correlate them (lower $\rho$)**, which is exactly why Random Forest adds "feature randomness" on top of bagging (8.2).


<a id="3"></a>
## 3. 数据 + 从零实现 ⭐ / Data & From Scratch

用 **Titanic**（5.6 清洗过）。从零实现 bagging：循环 B 次，每次 bootstrap 一个样本、训一棵**深树**（高方差基学习器），预测时**多数投票**。
Using **Titanic** (cleaned in 5.6). Bagging from scratch: loop B times, bootstrap a sample, train a **deep tree** (a high-variance base learner) each time, and **majority-vote** at prediction.


In [ ]:
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from scipy.stats import mode

df = sns.load_dataset("titanic")
feat = ["pclass", "sex", "age", "sibsp", "parch", "fare"]
d = df[feat + ["survived"]].copy()
d["age"] = d["age"].fillna(d["age"].median())
d["fare"] = d["fare"].fillna(d["fare"].median())
d["sex"] = (d["sex"] == "male").astype(int)
X, y = d[feat].values, d["survived"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

class BaggingScratch:
    def __init__(self, n_estimators=100, seed=0):
        self.n_estimators, self.seed = n_estimators, seed
    def fit(self, X, y):
        rng = np.random.default_rng(self.seed); n = len(X)
        self.trees = []
        for _ in range(self.n_estimators):
            idx = rng.integers(0, n, n)                  # bootstrap: 有放回抽 n 个索引
            # 每棵都是不剪枝的深树(高方差基学习器), 在各自 bootstrap 上训 / deep tree per bootstrap
            self.trees.append(DecisionTreeClassifier(random_state=0).fit(X[idx], y[idx]))
        return self
    def predict(self, X):
        # 收集所有树的预测(B×n), 对每个样本取多数票 / collect votes, take majority
        preds = np.array([t.predict(X) for t in self.trees])
        return mode(preds, axis=0, keepdims=False).mode

bag = BaggingScratch(n_estimators=100).fit(X_tr, y_tr)
print(f"从零 bagging(100 棵深树) test 准确率: {(bag.predict(X_te) == y_te).mean():.3f}")

from sklearn.ensemble import BaggingClassifier
sk = BaggingClassifier(DecisionTreeClassifier(random_state=0), n_estimators=100, random_state=0).fit(X_tr, y_tr)
print(f"sklearn BaggingClassifier   test 准确率: {sk.score(X_te, y_te):.3f}")
print(f"单棵深树(对照)              test 准确率: {DecisionTreeClassifier(random_state=0).fit(X_tr,y_tr).score(X_te,y_te):.3f}")
print("→ bagging 明显优于单棵树 — 投票抵消了单树的高方差")


<a id="4"></a>
## 4. 谁适合 bagging：高方差才有效 ⭐ / Who Benefits: High-Variance Only

一个常被忽略的关键点（面试加分）：**bagging 只对高方差、不稳定的基学习器有效**（如深树）。对**低方差、稳定的模型**（如逻辑回归、浅树），每个 bootstrap 训出来的模型几乎一样（$\rho\approx1$），平均了也没用——回顾公式，$\rho\approx1$ 时方差降不下来。
An often-missed key point (interview bonus): **bagging helps only high-variance, unstable base learners** (like deep trees). For **low-variance, stable models** (logistic regression, shallow trees), every bootstrap yields nearly the same model ($\rho\approx1$), so averaging does nothing — recall the formula: with $\rho\approx1$ the variance won't drop.

下面对比"bagging 深树"和"bagging 逻辑回归"各自相对单模型的提升。
Below we compare the lift from bagging deep trees vs bagging logistic regression, each relative to its single model.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

base_learners = {
    "深树 deep tree (高方差)": DecisionTreeClassifier(random_state=0),
    "逻辑回归 logreg (低方差)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
}
print(f"{'基学习器':<26}{'单模型':>9}{'bagging':>9}{'提升':>8}")
for name, base in base_learners.items():
    single = cross_val_score(base, X, y, cv=5).mean()
    bagged = cross_val_score(BaggingClassifier(base, n_estimators=50, random_state=0), X, y, cv=5).mean()
    print(f"{name:<28}{single:>9.3f}{bagged:>9.3f}{bagged-single:>+8.3f}")
print("\n深树: bagging 提升明显(高方差→平均有用); 逻辑回归: 几乎无提升(低方差→平均无用)")
print("→ bagging 的基学习器要选'高方差不稳定'的(深树是经典选择)")


<a id="5"></a>
## 5. OOB 免费验证 + 小结 ⭐ / OOB & Summary

第 1 节说每个 bootstrap 留下约 37% 袋外(OOB)样本。用每个样本"没训练过它的那些树"来预测它，汇总就得到一个**几乎免费的泛化估计**——不用单独划验证集。设 `oob_score=True` 即可。
Section 1 noted each bootstrap leaves ~37% out-of-bag (OOB) samples. Predicting each sample with only the trees that never trained on it gives a **nearly free generalization estimate** — no separate validation set. Just set `oob_score=True`.


In [ ]:
bag_oob = BaggingClassifier(DecisionTreeClassifier(random_state=0), n_estimators=200,
                            oob_score=True, random_state=0).fit(X_tr, y_tr)
print(f"OOB 准确率(免费, 没用 test): {bag_oob.oob_score_:.3f}")
print(f"真正的 test 准确率:          {bag_oob.score(X_te, y_te):.3f}")
print(f"5-fold CV 准确率:           {cross_val_score(BaggingClassifier(DecisionTreeClassifier(random_state=0), n_estimators=100, random_state=0), X_tr, y_tr, cv=5).mean():.3f}")
print("OOB ≈ test ≈ CV → OOB 是可靠的泛化估计, 零额外成本(训练时顺便算出)")


```
bagging: 同模型在 B 个 bootstrap 样本上各训一个, 投票(分类)/平均(回归); 降方差
bootstrap: 有放回抽 n 个, 约覆盖 63.2% 原始样本(1-1/e), 剩 36.8% 袋外(OOB)
降方差数学: Var = ρσ² + (1-ρ)/B σ²; 加模型消第二项, 但被 ρ 卡住 → 还要去相关(→RF 8.2)
只降方差不降偏差; 只对高方差/不稳定基学习器有效(深树✓, 逻辑回归✗)
OOB: 用没训过某样本的树预测它, 免费泛化估计, ≈test≈CV
bagging(并行降方差) vs boosting(串行降偏差)
```

### 💡 面试速查 / Interview cheat-sheet
1. **bagging 降方差**: 平均/投票 B 个 bootstrap 训出的模型。
   Bagging reduces variance by averaging/voting B bootstrap-trained models.
2. **bootstrap 约覆盖 63%**(1-1/e), 剩 37% 是 OOB。
   A bootstrap covers ~63% (1-1/e), leaving ~37% OOB.
3. **降方差公式 ρσ²+(1-ρ)σ²/B**: 加模型 + 去相关都重要。
   The formula ρσ²+(1-ρ)σ²/B: both more models and de-correlation matter.
4. **只对高方差基学习器有效**(深树); 低方差模型无用。
   Helps only high-variance learners (deep trees); useless for low-variance ones.
5. **OOB ≈ 免费 CV**; bagging 并行降方差(对比 boosting 串行降偏差)。
   OOB ≈ free CV; bagging is parallel variance reduction (vs boosting's sequential bias reduction).

### 下一节 / Next
**8.2 随机森林深入**——在 bagging 基础上加"特征随机"去相关(降 ρ), 系统讲 OOB、特征重要性的偏差与置换重要性。
**8.2 Random Forest Deep Dive** — adds "feature randomness" on top of bagging to de-correlate (lower ρ); deep dive into OOB, importance bias, and permutation importance.
